# CMT307 Applied Machine Learning — ASHRAE Great Energy Predictor III

**Cardiff University | Spring 2025/26 | Task 9: Energy Usage Prediction**

---

| | |
|---|---|
| **Name** | Zahra |
| **Role** | Person 4 — Data Merging & Integration Hub |
| **Notebook** | `Zahra_Data_Merging.ipynb` |

In [1]:
import pandas as pd
import os

DATA_DIR = '../data/ashrae-energy-prediction'

train        = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'), parse_dates=['timestamp'])
metadata     = pd.read_csv(os.path.join(DATA_DIR, 'building_metadata.csv'))
weather      = pd.read_csv(os.path.join(DATA_DIR, 'weather_train.csv'), parse_dates=['timestamp'])
test         = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'), parse_dates=['timestamp'])
weather_test = pd.read_csv(os.path.join(DATA_DIR, 'weather_test.csv'), parse_dates=['timestamp'])

print('train:       ', train.shape)
print('metadata:    ', metadata.shape)
print('weather:     ', weather.shape)
print('test:        ', test.shape)
print('weather_test:', weather_test.shape)

train:        (20216100, 4)
metadata:     (1449, 6)
weather:      (139773, 9)
test:         (41697600, 4)
weather_test: (277243, 9)


In [2]:
# Check all building_ids in train exist in metadata
missing_ids = set(train['building_id']) - set(metadata['building_id'])
print(f'building_ids in train not in metadata: {len(missing_ids)}')
print(f'unique buildings in train:    {train["building_id"].nunique()}')
print(f'unique buildings in metadata: {metadata["building_id"].nunique()}')

building_ids in train not in metadata: 0
unique buildings in train:    1449
unique buildings in metadata: 1449


In [3]:
# Merge train + metadata on building_id
merged_train = train.merge(metadata, on='building_id', how='left')
print('After train + metadata:', merged_train.shape)  # rows must stay at 20,216,100

# Merge + weather on site_id + timestamp
merged_train = merged_train.merge(weather, on=['site_id', 'timestamp'], how='left')
print('After + weather:       ', merged_train.shape)  # must still be 20,216,100

After train + metadata: (20216100, 9)
After + weather:        (20216100, 16)


In [4]:
# Null counts after merge
nulls = merged_train.isnull().sum()
print(nulls[nulls > 0])

year_built            12127645
floor_count           16709167
air_temperature          96658
cloud_coverage         8825365
dew_temperature         100140
precip_depth_1_hr      3749023
sea_level_pressure     1231669
wind_direction         1449048
wind_speed              143676
dtype: int64


In [5]:
# Repeat for test
merged_test = test.merge(metadata, on='building_id', how='left')
merged_test = merged_test.merge(weather_test, on=['site_id', 'timestamp'], how='left')
print('merged_test:', merged_test.shape)

nulls_test = merged_test.isnull().sum()
print(nulls_test[nulls_test > 0])

merged_test: (41697600, 16)
year_built            24598080
floor_count           34444320
air_temperature         221901
cloud_coverage        19542180
dew_temperature         260799
precip_depth_1_hr      7801563
sea_level_pressure     2516826
wind_direction         2978663
wind_speed              302089
dtype: int64


In [6]:
# Save merged files as CSV
os.makedirs('../data_processed', exist_ok=True)
merged_train.to_csv('../data_processed/merged_train.csv', index=False)
merged_test.to_csv('../data_processed/merged_test.csv', index=False)
print('Saved merged_train.csv and merged_test.csv to data_processed/')

Saved merged_train.csv and merged_test.csv to data_processed/


### Key Findings

- All 5 files loaded successfully: train (20,216,100 rows), metadata (1,449), weather_train (139,773), test (41,697,600), weather_test (277,243)
- All 1,449 building IDs in train are present in metadata — no orphan records
- Both merges preserved row count exactly (20,216,100 for train, 41,697,600 for test) — no row duplication or loss
- 16 columns in final merged tables (4 original + 5 metadata + 7 weather features)
- Columns with missing values after merge:
  - `year_built`: 60% missing — sparse in source metadata
  - `floor_count`: 83% missing — sparse in source metadata
  - `cloud_coverage`: 44% missing — weather station gaps
  - `precip_depth_1_hr`: 19% missing
  - `sea_level_pressure`: 6% missing
  - `wind_direction`: 7% missing
  - `air_temperature`, `dew_temperature`, `wind_speed`: <1% missing
- Missing weather values are due to gaps in the original weather station recordings, not a merge error
- Saved: `merged_train.csv` and `merged_test.csv` to `data_processed/`

---

## Sprint 2 — Data Merging, Preprocessing & Feature Engineering

---

## Sprint 3 — Preprocessing Pipeline & Train/Val Split

---

## Sprint 4 — Evaluation & Report